In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/processed/st12000nm0008_2026q1q2_clean.csv.gz",
                   parse_dates=["date"])
fail_dates = pd.read_csv("../data/processed/fail_dates.csv", parse_dates=["fail_date"])

failed_serials = fail_dates["serial_number"].unique()
all_serials = data["serial_number"].unique()
healthy_serials = np.setdiff1d(all_serials, failed_serials)

print("total drives:", len(all_serials), "| failed:", len(failed_serials), "| healthy:", len(healthy_serials))

rng = np.random.default_rng(42)

def split_serials(serials, train_frac=0.7, val_frac=0.15):
    serials = rng.permutation(serials)
    n = len(serials)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    return serials[:n_train], serials[n_train:n_train+n_val], serials[n_train+n_val:]

f_train, f_val, f_test = split_serials(failed_serials)
h_train, h_val, h_test = split_serials(healthy_serials)

train_serials = np.concatenate([f_train, h_train])
val_serials   = np.concatenate([f_val, h_val])
test_serials  = np.concatenate([f_test, h_test])

print("train:", len(train_serials), "(failed:", len(f_train), ")")
print("val:  ", len(val_serials),   "(failed:", len(f_val), ")")
print("test: ", len(test_serials),  "(failed:", len(f_test), ")")

split_map = pd.DataFrame({
    "serial_number": np.concatenate([train_serials, val_serials, test_serials]),
    "split": ["train"]*len(train_serials) + ["val"]*len(val_serials) + ["test"]*len(test_serials)
})
split_map.to_csv("../data/processed/drive_split.csv", index=False)
print(split_map["split"].value_counts())

total drives: 18652 | failed: 274 | healthy: 18378
train: 13055 (failed: 191 )
val:   2797 (failed: 41 )
test:  2800 (failed: 42 )
split
train    13055
test      2800
val       2797
Name: count, dtype: int64
